In [0]:
%run ./00_config

### Standardize the raw taxi zone lookup into a clean dimension table

In [0]:
from pyspark.sql import functions as F

# Source: nyc_taxi_company.bronze.taxi_zones
# Target: nyc_taxi_company.silver.taxi_zones
# Physical Delta path: abfss://taxicompany@azinfrasummit.dfs.core.windows.net/curated/silver/taxi_zones

bronze_zone_table = f"{BRONZE_SCHEMA}.taxi_zones"
silver_zone_table = f"{SILVER_SCHEMA}.taxi_zones"
silver_zone_path = f"{CURATED_PATH}/silver/taxi_zones"

print(f"Reading Bronze table: {bronze_zone_table}")
print(f"Writing Silver table: {silver_zone_table}")
print(f"Silver Delta path: {silver_zone_path}")

In [0]:
zones_raw_df = spark.table(bronze_zone_table)

display(zones_raw_df.limit(10))
zones_raw_df.printSchema()

In [0]:
#create a silver dataframe for taxi zones

zones_silver_df = (
    zones_raw_df
    .select(
        F.col("LocationID").cast("int").alias("location_id"),
        F.trim(F.col("Borough")).alias("borough"),
        F.trim(F.col("Zone")).alias("zone_name"),
        F.trim(F.col("service_zone")).alias("service_zone")
    )
    .withColumn("is_airport", F.col("zone_name").isin("JFK Airport", "LaGuardia Airport", "Newark Airport"))
    .withColumn("is_manhattan", F.col("borough") == F.lit("Manhattan"))
    .withColumn("_silver_updated_ts", F.current_timestamp())
)

display(zones_silver_df.orderBy("location_id").limit(20))

In [0]:
#write the delta table to ADLS and register the external location on UC
(
    zones_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_zone_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_zone_table}
USING DELTA
LOCATION '{silver_zone_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {silver_zone_table} IS
'Cleaned taxi zone dimension table mapping TLC LocationID values to borough, zone name, service zone, and helper flags.'
""")

print(f"Registered or confirmed table: {silver_zone_table}")

In [0]:
# Silver taxi zones validation

display(spark.sql(f"""
SELECT *
FROM {silver_zone_table}
ORDER BY location_id
LIMIT 20
"""))

display(spark.sql(f"""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT location_id) AS distinct_location_ids
FROM {silver_zone_table}
"""))

display(spark.sql(f"""
SELECT
  borough,
  service_zone,
  COUNT(*) AS zone_count
FROM {silver_zone_table}
GROUP BY borough, service_zone
ORDER BY borough, service_zone
"""))

### Clean NYC Yellow Taxi trip records and enrich them with pickup/dropoff zone metadata

In [0]:
# paths for ingestion
# Source:
#   nyc_taxi_company.bronze.yellow_taxi_trips
#   nyc_taxi_company.silver.taxi_zones
# Target:
#   nyc_taxi_company.silver.trips_clean

bronze_taxi_table = f"{BRONZE_SCHEMA}.yellow_taxi_trips"
silver_zone_table = f"{SILVER_SCHEMA}.taxi_zones"

silver_trips_table = f"{SILVER_SCHEMA}.trips_clean"
silver_trips_path = f"{CURATED_PATH}/silver/trips_clean"

print(f"Reading Bronze taxi trips: {bronze_taxi_table}")
print(f"Reading Silver taxi zones: {silver_zone_table}")
print(f"Writing Silver trips table: {silver_trips_table}")
print(f"Silver trips path: {silver_trips_path}")

In [0]:
trips_raw_df = spark.table(bronze_taxi_table)
zones_df = spark.table(silver_zone_table)

display(trips_raw_df.limit(10))
trips_raw_df.printSchema()

In [0]:
# create cleaned silver dataframe

trips_base_df = (
    trips_raw_df
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.col("VendorID").cast("string"),
                F.col("tpep_pickup_datetime").cast("string"),
                F.col("tpep_dropoff_datetime").cast("string"),
                F.col("PULocationID").cast("string"),
                F.col("DOLocationID").cast("string"),
                F.col("trip_distance").cast("string"),
                F.col("total_amount").cast("string")
            ),
            256
        ).alias("trip_id"),

        F.col("VendorID").cast("int").alias("vendor_id"),
        F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_ts"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_ts"),

        F.to_date(F.col("tpep_pickup_datetime")).alias("pickup_date"),
        F.hour(F.col("tpep_pickup_datetime")).alias("pickup_hour"),
        F.dayofweek(F.col("tpep_pickup_datetime")).alias("pickup_day_of_week"),
        F.month(F.col("tpep_pickup_datetime")).alias("pickup_month"),

        F.col("PULocationID").cast("int").alias("pickup_location_id"),
        F.col("DOLocationID").cast("int").alias("dropoff_location_id"),

        F.col("passenger_count").cast("int").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance_miles"),

        F.col("RatecodeID").cast("int").alias("rate_code_id"),
        F.col("payment_type").cast("int").alias("payment_type"),

        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("extra").cast("double").alias("extra_amount"),
        F.col("mta_tax").cast("double").alias("mta_tax_amount"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("tolls_amount").cast("double").alias("tolls_amount"),
        F.col("improvement_surcharge").cast("double").alias("improvement_surcharge"),
        F.col("total_amount").cast("double").alias("total_amount"),

        # Hive partition columns from Auto Loader (year=YYYY/month=MM directory structure)
        F.col("year").alias("_source_year"),
        F.col("month").alias("_source_month"),
        F.col("_source_file"),
        F.col("_ingest_ts")
    )
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / F.lit(60.0)
    )
    .withColumn(
        "avg_speed_mph",
        F.when(
            F.col("trip_duration_minutes") > 0,
            F.col("trip_distance_miles") / (F.col("trip_duration_minutes") / F.lit(60.0))
        )
    )
    .withColumn("_silver_updated_ts", F.current_timestamp())
)

display(trips_base_df.limit(10))

In [0]:
# Apply quality filters
# Date range: 2020-01-01 through end of current month (dynamic)
# All other quality filters (duration, distance, amount) unchanged.

from datetime import datetime
from dateutil.relativedelta import relativedelta

now = datetime.now()
first_of_next_month = (now.replace(day=1) + relativedelta(months=1)).strftime("%Y-%m-%d 00:00:00")

SILVER_TRIPS_START_TS = "2020-01-01 00:00:00"
SILVER_TRIPS_END_TS = first_of_next_month

print(f"Trips date filter: {SILVER_TRIPS_START_TS} to {SILVER_TRIPS_END_TS}")

trips_filtered_df = (
    trips_base_df
    .filter(F.col("pickup_ts").isNotNull())
    .filter(F.col("dropoff_ts").isNotNull())
    .filter(F.col("pickup_ts") >= F.to_timestamp(F.lit(SILVER_TRIPS_START_TS)))
    .filter(F.col("pickup_ts") < F.to_timestamp(F.lit(SILVER_TRIPS_END_TS)))
    .filter(F.col("dropoff_ts") > F.col("pickup_ts"))
    .filter(F.col("pickup_location_id").isNotNull())
    .filter(F.col("dropoff_location_id").isNotNull())
    .filter(F.col("trip_distance_miles") > 0)
    .filter(F.col("trip_duration_minutes").between(1, 240))
    .filter(F.col("total_amount") >= 0)
)

print(f"Raw trips count: {trips_base_df.count():,}")
print(f"Filtered trips count: {trips_filtered_df.count():,}")

In [0]:
# join pickup and dropoff zone metadata

pickup_zones_df = (
    zones_df
    .select(
        F.col("location_id").alias("pickup_location_id"),
        F.col("borough").alias("pickup_borough"),
        F.col("zone_name").alias("pickup_zone"),
        F.col("service_zone").alias("pickup_service_zone"),
        F.col("is_airport").alias("pickup_is_airport"),
        F.col("is_manhattan").alias("pickup_is_manhattan")
    )
)

dropoff_zones_df = (
    zones_df
    .select(
        F.col("location_id").alias("dropoff_location_id"),
        F.col("borough").alias("dropoff_borough"),
        F.col("zone_name").alias("dropoff_zone"),
        F.col("service_zone").alias("dropoff_service_zone"),
        F.col("is_airport").alias("dropoff_is_airport"),
        F.col("is_manhattan").alias("dropoff_is_manhattan")
    )
)

trips_clean_df = (
    trips_filtered_df
    .join(pickup_zones_df, on="pickup_location_id", how="left")
    .join(dropoff_zones_df, on="dropoff_location_id", how="left")
)

display(trips_clean_df.limit(10))

In [0]:
#write dataframe as a delta table to ADLS and register the external location on UC

(
    trips_clean_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_trips_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_trips_table}
USING DELTA
LOCATION '{silver_trips_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {silver_trips_table} IS
'Cleaned and zone-enriched NYC Yellow Taxi trips derived from Bronze trip records and Silver taxi zone lookup.'
""")

print(f"Registered or confirmed table: {silver_trips_table}")

In [0]:
# Validation checks

display(spark.sql(f"""
SELECT COUNT(*) AS row_count
FROM {silver_trips_table}
"""))

# Check for any rows outside the expected date range
display(spark.sql(f"""
SELECT
  pickup_date,
  COUNT(*) AS row_count
FROM {silver_trips_table}
WHERE pickup_date < DATE '2020-01-01'
   OR pickup_date >= DATE '{SILVER_TRIPS_END_TS[:10]}'
GROUP BY pickup_date
ORDER BY pickup_date
"""))

display(spark.sql(f"""
SELECT
  MIN(pickup_ts) AS min_pickup_ts,
  MAX(pickup_ts) AS max_pickup_ts,
  MIN(dropoff_ts) AS min_dropoff_ts,
  MAX(dropoff_ts) AS max_dropoff_ts
FROM {silver_trips_table}
"""))

display(spark.sql(f"""
SELECT
  pickup_borough,
  COUNT(*) AS trip_count
FROM {silver_trips_table}
GROUP BY pickup_borough
ORDER BY trip_count DESC
"""))

display(spark.sql(f"""
SELECT
  pickup_zone,
  pickup_borough,
  COUNT(*) AS trip_count
FROM {silver_trips_table}
GROUP BY pickup_zone, pickup_borough
ORDER BY trip_count DESC
LIMIT 20
"""))

display(spark.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN pickup_borough IS NULL THEN 1 ELSE 0 END) AS missing_pickup_zone_rows,
  SUM(CASE WHEN dropoff_borough IS NULL THEN 1 ELSE 0 END) AS missing_dropoff_zone_rows
FROM {silver_trips_table}
"""))   

display(spark.sql(f"""
DESCRIBE DETAIL {silver_trips_table}
"""))

### Convert NOAA long-format daily observations into one row per date with usable weather fields


In [0]:
#paths for silver ingestion
# Source: nyc_taxi_company.bronze.weather_daily
# Target: nyc_taxi_company.silver.weather_daily

bronze_weather_table = f"{BRONZE_SCHEMA}.weather_daily"
silver_weather_table = f"{SILVER_SCHEMA}.weather_daily"
silver_weather_path = f"{CURATED_PATH}/silver/weather_clean"

print(f"Reading Bronze weather table: {bronze_weather_table}")
print(f"Writing Silver weather table: {silver_weather_table}")
print(f"Silver weather path: {silver_weather_path}")

In [0]:
#load silver table

weather_raw_df = spark.table(bronze_weather_table)

display(weather_raw_df.limit(20))
weather_raw_df.printSchema()

In [0]:
#silver transformations

weather_pivot_df = (
    weather_raw_df
    .select(
        F.to_date("date").alias("weather_date"),
        F.col("station").alias("station_id"),
        F.col("datatype"),
        F.col("value").cast("double").alias("value")
    )
    .groupBy("weather_date", "station_id")
    .pivot("datatype", ["PRCP", "SNOW", "TMAX", "TMIN"])
    .agg(F.first("value"))
)

weather_silver_df = (
    weather_pivot_df
    .withColumn("station_name", F.lit("NY CITY CENTRAL PARK, NY US"))

    # NOAA CDO values with units=metric:
    # PRCP and SNOW are in mm.
    # TMAX and TMIN are in degrees C for this API response when units=metric.
    .withColumn("precipitation_mm", F.coalesce(F.col("PRCP"), F.lit(0.0)))
    .withColumn("snowfall_mm", F.coalesce(F.col("SNOW"), F.lit(0.0)))
    .withColumn("temp_max_c", F.col("TMAX"))
    .withColumn("temp_min_c", F.col("TMIN"))
    .withColumn("temp_avg_c", (F.col("temp_max_c") + F.col("temp_min_c")) / F.lit(2.0))

    .withColumn("precipitation_inches", F.col("precipitation_mm") / F.lit(25.4))
    .withColumn("snowfall_inches", F.col("snowfall_mm") / F.lit(25.4))
    .withColumn("temp_max_f", F.col("temp_max_c") * F.lit(9.0 / 5.0) + F.lit(32.0))
    .withColumn("temp_min_f", F.col("temp_min_c") * F.lit(9.0 / 5.0) + F.lit(32.0))
    .withColumn("temp_avg_f", F.col("temp_avg_c") * F.lit(9.0 / 5.0) + F.lit(32.0))

    .withColumn("has_precipitation", F.col("precipitation_inches") > F.lit(0))
    .withColumn("has_snow", F.col("snowfall_inches") > F.lit(0))

    .withColumn(
        "weather_condition",
        F.when(F.col("has_snow"), F.lit("Snow"))
         .when(F.col("has_precipitation"), F.lit("Rain"))
         .when(F.col("temp_avg_f") < F.lit(32), F.lit("Cold"))
         .when(F.col("temp_avg_f") >= F.lit(85), F.lit("Hot"))
         .otherwise(F.lit("Clear/Other"))
    )
    .select(
        "weather_date",
        "station_id",
        "station_name",
        "precipitation_mm",
        "precipitation_inches",
        "snowfall_mm",
        "snowfall_inches",
        "temp_max_c",
        "temp_min_c",
        "temp_avg_c",
        "temp_max_f",
        "temp_min_f",
        "temp_avg_f",
        "has_precipitation",
        "has_snow",
        "weather_condition"
    )
    .withColumn("_silver_updated_ts", F.current_timestamp())
)

display(weather_silver_df.orderBy("weather_date"))


In [0]:
# write dataframe as a delta table in ADLS and register external location on UC

(
    weather_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_weather_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_weather_table}
USING DELTA
LOCATION '{silver_weather_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {silver_weather_table} IS
'Cleaned daily weather table for NYC Central Park station, converted from NOAA Daily Summary observations.'
""")

print(f"Registered or confirmed table: {silver_weather_table}")

In [0]:
#validations

display(spark.sql(f"""
SELECT
  weather_condition,
  COUNT(*) AS day_count,
  ROUND(AVG(precipitation_inches), 3) AS avg_precipitation_inches,
  ROUND(AVG(snowfall_inches), 3) AS avg_snowfall_inches,
  ROUND(AVG(temp_avg_f), 1) AS avg_temp_f
FROM {silver_weather_table}
GROUP BY weather_condition
ORDER BY day_count DESC
"""))

### Clean and standardize curated sports event reference data

In [0]:
# ingestion paths
# Source:
#   nyc_taxi_company.bronze.sports_events_raw
# Target:
#   nyc_taxi_company.silver.sports_events

bronze_sports_events_table = f"{BRONZE_SCHEMA}.sports_events_raw"
silver_sports_events_table = f"{SILVER_SCHEMA}.sports_events"
silver_sports_events_path = f"{CURATED_PATH}/silver/sports_events"

print(f"Reading Bronze sports events table: {bronze_sports_events_table}")
print(f"Writing Silver sports events table: {silver_sports_events_table}")
print(f"Silver sports events path: {silver_sports_events_path}")

In [0]:
#validate 

sports_events_raw_df = spark.table(bronze_sports_events_table)

display(sports_events_raw_df.orderBy("event_start_ts_local"))
sports_events_raw_df.printSchema()

In [0]:
#silver transformations

# COMMAND ----------

sports_events_silver_df = (
    sports_events_raw_df
    .select(
        F.col("event_id").cast("string").alias("event_id"),
        F.col("event_source").cast("string").alias("event_source"),
        F.col("league").cast("string").alias("league"),
        F.col("season").cast("int").alias("season"),
        F.col("team_name").cast("string").alias("team_name"),
        F.col("opponent_name").cast("string").alias("opponent_name"),
        F.col("is_home_event").cast("boolean").alias("is_home_event"),
        F.col("event_name").cast("string").alias("event_name"),
        F.col("event_type").cast("string").alias("event_type"),
        F.col("venue_name").cast("string").alias("venue_name"),
        F.col("venue_borough").cast("string").alias("venue_borough"),
        F.col("venue_taxi_zone_hint").cast("string").alias("venue_taxi_zone_hint"),
        F.col("venue_pickup_location_id").cast("int").alias("venue_pickup_location_id"),
        F.col("venue_latitude").cast("double").alias("venue_latitude"),
        F.col("venue_longitude").cast("double").alias("venue_longitude"),

        # These strings include offsets. Spark can parse them into timestamps.
        F.to_timestamp("event_start_ts_utc").alias("event_start_ts_utc"),
        F.to_timestamp("event_start_ts_local").alias("event_start_ts_local"),
        F.to_date("event_date_local").alias("event_date_local"),
        F.col("event_hour_local").cast("int").alias("event_hour_local"),
        F.to_timestamp("estimated_event_end_ts_local").alias("estimated_event_end_ts_local"),
        F.col("event_duration_hours").cast("double").alias("event_duration_hours"),

        F.col("analysis_before_hours").cast("int").alias("analysis_before_hours"),
        F.col("analysis_after_hours").cast("int").alias("analysis_after_hours"),
        F.col("include_in_taxi_analysis").cast("boolean").alias("include_in_taxi_analysis"),
        F.col("source_url").cast("string").alias("source_url"),

        F.col("_source_file"),
        F.col("_ingest_ts")
    )
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("event_start_ts_local").isNotNull())
    .filter(F.col("estimated_event_end_ts_local").isNotNull())
    .filter(F.col("is_home_event") == F.lit(True))
    .filter(F.col("include_in_taxi_analysis") == F.lit(True))
    .withColumn("event_start_date_local", F.to_date("event_start_ts_local"))
    .withColumn("event_start_hour_local", F.hour("event_start_ts_local"))
    .withColumn("estimated_event_end_hour_local", F.hour("estimated_event_end_ts_local"))
    .withColumn(
        "analysis_window_start_ts",
        F.expr("event_start_ts_local - INTERVAL 4 HOURS")
    )
    .withColumn(
        "analysis_window_end_ts",
        F.expr("estimated_event_end_ts_local + INTERVAL 4 HOURS")
    )
    .withColumn("_silver_updated_ts", F.current_timestamp())
)

display(sports_events_silver_df.orderBy("event_start_ts_local"))

In [0]:
#write dataframe to delta table in ADLS and register external location on UC 

(
    sports_events_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_sports_events_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_sports_events_table}
USING DELTA
LOCATION '{silver_sports_events_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {silver_sports_events_table} IS
'Cleaned sports event reference table for NYC-area home games used for taxi event-demand analysis.'
""")

print(f"Registered or confirmed table: {silver_sports_events_table}")

In [0]:
#validations

display(spark.sql(f"""
SELECT COUNT(*) AS row_count
FROM {silver_sports_events_table}
"""))

display(spark.sql(f"""
SELECT
  league,
  team_name,
  venue_name,
  COUNT(*) AS event_count,
  MIN(event_start_ts_local) AS first_event,
  MAX(event_start_ts_local) AS last_event
FROM {silver_sports_events_table}
GROUP BY league, team_name, venue_name
ORDER BY league, team_name
"""))

display(spark.sql(f"""
SELECT
  event_id,
  league,
  team_name,
  opponent_name,
  venue_name,
  event_start_ts_local,
  estimated_event_end_ts_local,
  analysis_window_start_ts,
  analysis_window_end_ts
FROM {silver_sports_events_table}
ORDER BY event_start_ts_local
"""))

### Curate zone catchment data

In [0]:
%sql
SELECT location_id, borough, zone_name, service_zone
FROM nyc_taxi_company.silver.taxi_zones
WHERE lower(zone_name) LIKE '%penn%'
   OR lower(zone_name) LIKE '%madison%'
   OR lower(zone_name) LIKE '%midtown%'
   OR lower(zone_name) LIKE '%chelsea%'
   OR lower(zone_name) LIKE '%prospect%'
   OR lower(zone_name) LIKE '%park slope%'
   OR lower(zone_name) LIKE '%downtown brooklyn%'
   OR lower(zone_name) LIKE '%yankee%'
   OR lower(zone_name) LIKE '%highbridge%'
   OR lower(zone_name) LIKE '%flushing%'
   OR lower(zone_name) LIKE '%corona%'
   OR lower(zone_name) LIKE '%willets%'
   OR lower(zone_name) LIKE '%queensboro%'
ORDER BY borough, zone_name;

In [0]:
%sql
SELECT location_id, borough, zone_name, service_zone
FROM nyc_taxi_company.silver.taxi_zones
WHERE borough IN ('Manhattan', 'Brooklyn', 'Bronx', 'Queens')
ORDER BY borough, zone_name;

In [0]:
# build venue_zone_catchment
# Purpose:
# Map sports venues to one or more nearby TLC taxi zones for event demand analysis.
# Catchment design:
#   primary = best single zone for the venue
#   nearby  = adjacent/spillover zones where event-related pickup/dropoff demand may appear
#   For Yankee Stadium, West Concourse is the primary zone, with neighboring Bronx zones included as spillover.”

venue_catchment_rows = [
    # Madison Square Garden / Penn Station area
    {"venue_name": "Madison Square Garden", "location_id": 186, "catchment_type": "primary"},
    {"venue_name": "Madison Square Garden", "location_id": 100, "catchment_type": "nearby"},
    {"venue_name": "Madison Square Garden", "location_id": 164, "catchment_type": "nearby"},
    {"venue_name": "Madison Square Garden", "location_id": 246, "catchment_type": "nearby"},

    # Barclays Center / Prospect Heights area
    {"venue_name": "Barclays Center", "location_id": 189, "catchment_type": "primary"},
    {"venue_name": "Barclays Center", "location_id": 97, "catchment_type": "nearby"},
    {"venue_name": "Barclays Center", "location_id": 65, "catchment_type": "nearby"},
    {"venue_name": "Barclays Center", "location_id": 181, "catchment_type": "nearby"},

    # Yankee Stadium / Bronx
    {"venue_name": "Yankee Stadium", "location_id": 247, "catchment_type": "primary"},
    {"venue_name": "Yankee Stadium", "location_id": 69, "catchment_type": "nearby"},
    {"venue_name": "Yankee Stadium", "location_id": 119, "catchment_type": "nearby"},

    # Citi Field / Queens
    {"venue_name": "Citi Field", "location_id": 93, "catchment_type": "primary"},
    {"venue_name": "Citi Field", "location_id": 253, "catchment_type": "nearby"},
    {"venue_name": "Citi Field", "location_id": 92, "catchment_type": "nearby"},
    {"venue_name": "Citi Field", "location_id": 173, "catchment_type": "nearby"},
]

In [0]:
# create silver df

venue_catchment_path = f"{CURATED_PATH}/silver/venue_zone_catchment"
venue_catchment_table = f"{SILVER_SCHEMA}.venue_zone_catchment"

zones_df = spark.table(f"{SILVER_SCHEMA}.taxi_zones")

venue_catchment_base_df = spark.createDataFrame(venue_catchment_rows)

venue_catchment_df = (
    venue_catchment_base_df
    .join(
        zones_df,
        on="location_id",
        how="left"
    )
    .select(
        "venue_name",
        "location_id",
        "borough",
        "zone_name",
        "service_zone",
        "catchment_type"
    )
    .withColumn("_silver_updated_ts", F.current_timestamp())
)

display(venue_catchment_df.orderBy("venue_name", "catchment_type", "zone_name"))

In [0]:
# write df to a delta table on ADLS and register the external location on UC

(
    venue_catchment_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(venue_catchment_path)
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {venue_catchment_table}
USING DELTA
LOCATION '{venue_catchment_path}'
""")

spark.sql(f"""
COMMENT ON TABLE {venue_catchment_table} IS
'Reference table mapping sports venues to nearby TLC taxi zones for event demand and origin-destination flow analysis.'
""")

print(f"Registered or confirmed table: {venue_catchment_table}")

In [0]:
#validations

# COMMAND ----------

display(spark.sql(f"""
SELECT *
FROM {venue_catchment_table}
ORDER BY venue_name, catchment_type, zone_name
"""))

display(spark.sql(f"""
SELECT
  venue_name,
  COUNT(*) AS mapped_zone_count,
  SUM(CASE WHEN location_id IS NULL OR zone_name IS NULL THEN 1 ELSE 0 END) AS missing_zone_rows
FROM {venue_catchment_table}
GROUP BY venue_name
ORDER BY venue_name
"""))